In [ ]:
# german_credit_fairness.ipynb

# Imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from fairlearn.metrics import MetricFrame, selection_rate, accuracy_score_group_min
from fairlearn.reductions import ExponentiatedGradient, DemographicParity
from fairlearn.reductions import GridSearch

# Load dataset (UCI German Credit)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"
columns = [
    "checking", "duration", "credit_history", "purpose", "credit_amount",
    "savings", "employment", "installment_rate", "personal_status",
    "other_debtors", "residence_since", "property", "age", "other_installment",
    "housing", "existing_credits", "job", "people_liable", "telephone",
    "foreign_worker", "target"
]
df = pd.read_csv(url, sep=" ", names=columns)

# Binary target: 1=Good, 0=Bad
df['target'] = df['target'].map({1: 1, 2: 0})

# Features and target
X = df.drop("target", axis=1)
y = df["target"]

# Sensitive attribute: Gender (extracted from 'personal_status')
# Simplify to Male vs Female
df['gender'] = df['personal_status'].apply(lambda x: 'male' if x in ['A91','A93','A94'] else 'female')
sensitive_feature = df['gender']

# Train-test split
X_train, X_test, y_train, y_test, s_train, s_test = train_test_split(
    X, y, sensitive_feature, test_size=0.3, random_state=42
)

# Preprocessing
categorical = X.select_dtypes(include=["object"]).columns
numeric = X.select_dtypes(exclude=["object"]).columns

preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("num", StandardScaler(), numeric)
])

# Pipeline
clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000))
])

# Train
clf.fit(X_train, y_train)

# Predictions
y_pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Fairness evaluation
metric_frame = MetricFrame(
    metrics={"accuracy": accuracy_score, "selection_rate": selection_rate},
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=s_test
)

print("By gender:\n", metric_frame.by_group)

metric_frame.by_group.plot(kind="bar", figsize=(8,5))
plt.title("Fairness metrics by gender")
plt.show()

# Mitigation using ExponentiatedGradient
constraint = DemographicParity()
mitigator = ExponentiatedGradient(clf, constraints=constraint)
mitigator.fit(X_train, y_train, sensitive_features=s_train)

y_pred_mitigated = mitigator.predict(X_test)

mf_mitigated = MetricFrame(
    metrics={"accuracy": accuracy_score, "selection_rate": selection_rate},
    y_true=y_test,
    y_pred=y_pred_mitigated,
    sensitive_features=s_test
)

print("Mitigated metrics:\n", mf_mitigated.by_group)
